## Cargar los datos ##

In [3]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('data/features_3_sec.csv', skipinitialspace=True)

# eliminar filas que no tienen género
df = df.dropna(subset=['label']) 

#feature que queremos predecir
y = df['label'] 

#features para el entrenamiento
X = df.drop(columns=["label"]).copy()

# Convertir columnas binarias de texto en 0 y 1
text_columns = X.select_dtypes(include=['str', 'object', 'string']).columns

for col in text_columns:
    values = X[col].dropna().unique()

    if len(values) == 2:
        X[col] = X[col].map({
            values[0]: 0,
            values[1]: 1})
            
#asegurar que solo hay valores numéricos
X = X.select_dtypes(include=np.number)

print("Número de canciones:", len(df))
print("Número de features:", X.shape[1])
print("\nCanciones por género:")
print(y.value_counts())

df.head()

Número de canciones: 9990
Número de features: 58

Canciones por género:
label
blues        1000
jazz         1000
metal        1000
pop          1000
reggae       1000
disco         999
classical     998
hiphop        998
rock          998
country       997
Name: count, dtype: int64


,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.0.wav,66149,0.335406,0.091048,0.130405,0.003521,1773.065032,167541.630869,1972.744388,117335.771563,...,39.687145,-3.241280,36.488243,0.722209,38.099152,-5.050335,33.618073,-0.243027,43.771767,blues
1,blues.00000.1.wav,66149,0.343065,0.086147,0.112699,0.001450,1816.693777,90525.690866,2010.051501,65671.875673,...,64.748276,-6.055294,40.677654,0.159015,51.264091,-2.837699,97.030830,5.784063,59.943081,blues
2,blues.00000.2.wav,66149,0.346815,0.092243,0.132003,0.004620,1788.539719,111407.437613,2084.565132,75124.921716,...,67.336563,-1.768610,28.348579,2.378768,45.717648,-1.938424,53.050835,2.517375,33.105122,blues
3,blues.00000.3.wav,66149,0.363639,0.086856,0.132565,0.002448,1655.289045,111952.284517,1960.039988,82913.639269,...,47.739452,-3.841155,28.337118,1.218588,34.770935,-3.580352,50.836224,3.630866,32.023678,blues
4,blues.00000.4.wav,66149,0.335579,0.088129,0.143289,0.001701,1630.656199,79667.267654,1948.503884,60204.020268,...,30.336359,0.664582,45.880913,1.689446,51.363583,-3.392489,26.738789,0.536961,29.146694,blues


## Definición de los modelos ##

Un pipeline es una cadena de pasos que se ejecutan en orden. Por ejemplo, antes de entrenar un modelo, podemos necesitar rellenar datos que faltan y normalizar las características.

`Pipeline` permite crear un pipeline combinando varios pasos

`SimpleImputer` sirve para rellenar los valores que faltan en los datos, por ejemplo, cuando una característica musical contiene NaN. se puede usar `StandardScaler` para rellenar esos datos vacíos

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [6]:
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000))
        ]),
        {
            "model__C": [0.01, 0.03, 0.1, 0.3, 1]
        }
    ),

    "k-NN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": [3, 5, 7, 9, 13, 17],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),

    "Decision Tree": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        {
            "model__max_depth": [5, 10, 15],
            "model__min_samples_split": [5, 10],
            "model__min_samples_leaf": [5, 10]
        }
    ),

    "Artificial Neural Network": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                max_iter=500,
                random_state=42,
                early_stopping=False,
                n_iter_no_change=15,
                tol=1e-3
            ))
        ]),
        {
            "model__hidden_layer_sizes": [(30,), (50,), (100,), (50, 25)],
            "model__alpha": [0.01, 0.1, 1.0]
        }
    )
}

## Definir la validación cruzada
 
Mismo esquema de 10 folds para comparar todos los algoritmos

In [7]:
from sklearn.model_selection import StratifiedKFold

In [8]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42)

## Entrenar y evaluar los modelos

Esta es la celda principal. `GridSearchCV` probará todas las configuraciones usando validación cruzada de 10 folds y seleccionará la que obtenga la mejor accuracy media para cada algoritmo

In [9]:
from sklearn.model_selection import GridSearchCV

In [10]:

results = []
best_models = {}

for name, (pipeline, params) in models.items():

    print("\nEvaluando:", name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1)

    search.fit(X, y)

    best_models[name] = search

    # Guardar los resultados de todas las configuraciones
    for i in range(len(search.cv_results_["params"])):
        results.append({
            "Algorithm": name,
            "Parameters": search.cv_results_["params"][i],
            "Train accuracy": search.cv_results_["mean_train_score"][i],
            "CV accuracy": search.cv_results_["mean_test_score"][i]})

    # Seleccionar la configuración con mayor accuracy
    # entre las que tienen overfitting < 0.15
    cv_results = pd.DataFrame(search.cv_results_)

    cv_results["Overfitting gap"] = (cv_results["mean_train_score"] - cv_results["mean_test_score"])

    valid_results = cv_results[cv_results["Overfitting gap"] < 0.15]

    #si no se alcanza el objetivo de ovefitting menor que 0,15, busca el que más ccuracy tenga directamente
    if not valid_results.empty:
        best_row = valid_results.sort_values("mean_test_score", ascending=False).iloc[0]
    else:
        best_row = cv_results.sort_values("mean_test_score", ascending=False).iloc[0]

    print("Best parameters:", best_row["params"])
    print("Best 10-fold CV accuracy (without overfitting):", round(best_row["mean_test_score"], 4))
    print("Overfitting gap:", round(best_row["Overfitting gap"], 4))
    print("Overfitting ok:", best_row["Overfitting gap"] < 0.15)


results_df = pd.DataFrame(results)

results_df["Overfitting gap"] = (
    results_df["Train accuracy"] -
    results_df["CV accuracy"]
)

results_df["Acceptable accuracy"] = (
    (results_df["CV accuracy"] > 0.70) 
)

results_df["Acceptable overfitting"] = (
    results_df["Overfitting gap"] < 0.15
)
results_df["Fits objectives"] = (
    (results_df["Overfitting gap"] < 0.15) & 
    (results_df["CV accuracy"] > 0.70)
)

results_df = results_df.sort_values(
    ["Acceptable overfitting", "CV accuracy", "Overfitting gap"],
    ascending=[False, False, True]
).reset_index(drop=True)

print(f"Total configs evaluated: {len(results_df)}")
print(f"Configs with appropiate overfitting: "
      f"{results_df['Acceptable overfitting'].sum()}")
print(f"Configs with both good accuracy and overfitting: "
      f"{results_df['Fits objectives'].sum()}")

display(results_df)



Evaluando: Logistic Regression
Best parameters: {'model__C': 1}
Best 10-fold CV accuracy (without overfitting): 0.7242
Overfitting gap: 0.0186
Overfitting ok: True

Evaluando: k-NN
Best parameters: {'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'distance'}
Best 10-fold CV accuracy (without overfitting): 0.9349
Overfitting gap: 0.0642
Overfitting ok: True

Evaluando: Decision Tree
Best parameters: {'model__max_depth': 15, 'model__min_samples_leaf': 10, 'model__min_samples_split': 10}
Best 10-fold CV accuracy (without overfitting): 0.6342
Overfitting gap: 0.1482
Overfitting ok: True

Evaluando: Artificial Neural Network
Best parameters: {'model__alpha': 0.1, 'model__hidden_layer_sizes': (100,)}
Best 10-fold CV accuracy (without overfitting): 0.8886
Overfitting gap: 0.1022
Overfitting ok: True
Total configs evaluated: 53
Configs with appropiate overfitting: 50
Configs with both good accuracy and overfitting: 39


,Algorithm,Parameters,Train accuracy,CV accuracy,Overfitting gap,Acceptable accuracy,Acceptable overfitting,Fits objectives
0,k-NN,"{'model__n_neighbors': 3, 'model__p': 1, 'mode...",0.999110,0.934935,0.064175,True,True,True
1,k-NN,"{'model__n_neighbors': 3, 'model__p': 1, 'mode...",0.970393,0.928529,0.041864,True,True,True
2,k-NN,"{'model__n_neighbors': 5, 'model__p': 1, 'mode...",0.999110,0.926727,0.072383,True,True,True
3,k-NN,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.999110,0.919319,0.079791,True,True,True
4,k-NN,"{'model__n_neighbors': 5, 'model__p': 1, 'mode...",0.957235,0.917117,0.040118,True,True,True
5,k-NN,"{'model__n_neighbors': 9, 'model__p': 1, 'mode...",0.999110,0.915516,0.083595,True,True,True
6,k-NN,"{'model__n_neighbors': 3, 'model__p': 2, 'mode...",0.999110,0.913113,0.085997,True,True,True
7,k-NN,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.943721,0.910310,0.033411,True,True,True
8,k-NN,"{'model__n_neighbors': 5, 'model__p': 2, 'mode...",0.999110,0.907708,0.091403,True,True,True
9,k-NN,"{'model__n_neighbors': 9, 'model__p': 1, 'mode...",0.932232,0.902603,0.029630,True,True,True
